In [1]:
import json
import asyncio
import sys
import sqlite3

from pathlib import Path

# Walk up from CWD to find the project root (identified by pyproject.toml),
# so imports work regardless of where Jupyter is launched from.
def find_project_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / "pyproject.toml").exists():
            return parent
    raise RuntimeError(f"Could not find project root from {start}")

project_root = find_project_root(Path().resolve())
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from movie_ingestion.metadata_generation.inputs import MovieInputData
from movie_ingestion.metadata_generation.generators.plot_events import generate_plot_events
from movie_ingestion.tracker import deserialize_imdb_row

/Users/michaelkeohane/Documents/movie-finder-rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ORIGINAL_SET_TMDB_IDS = [9377, 269149, 1584, 109445, 2493, 354912, 508965, 14160, 10674, 808, 13397, 76341, 85, 155, 245891, 1771, 569094, 299534, 11, 671, 120, 98, 27205, 603, 157336, 335984, 329, 329865, 493922, 694, 49018, 1034541, 176, 807, 496243, 419430, 1359, 550, 597, 13, 666277, 423, 11036, 1824, 25195, 216015, 392044, 545611, 22538, 37136]
MEDIUM_SPARSITY_TMDB_IDS = [329974, 1498832, 821937, 92, 160, 45739, 576560, 1383243, 1642210, 1639488]
HIGH_SPARSITY_TMDB_IDS = [270909, 493103, 64262, 1611977, 706910, 1297426, 35952, 158227, 215782, 1642486]

ALL_TMDB_IDS = ORIGINAL_SET_TMDB_IDS + MEDIUM_SPARSITY_TMDB_IDS + HIGH_SPARSITY_TMDB_IDS

# Open the tracker DB directly using the project root from the setup cell,
# avoiding reliance on CWD (init_db() uses a relative path).
db = sqlite3.connect(str(project_root / "ingestion_data" / "tracker.db"))
db.row_factory = sqlite3.Row

placeholders = ",".join("?" * len(ALL_TMDB_IDS))

# Fetch title + release_date from tmdb_data (the only fields tmdb_data has
# that we need; overview text lives in imdb_data.overview, not tmdb_data).
tmdb_rows: dict[int, sqlite3.Row] = {
    row["tmdb_id"]: row
    for row in db.execute(
        f"SELECT tmdb_id, title, release_date, maturity_rating "
        f"FROM tmdb_data WHERE tmdb_id IN ({placeholders})",
        ALL_TMDB_IDS,
    ).fetchall()
}

# Fetch all imdb_data columns; deserialize_imdb_row parses JSON array columns.
imdb_rows: dict[int, dict] = {
    row["tmdb_id"]: deserialize_imdb_row(row)
    for row in db.execute(
        f"SELECT * FROM imdb_data WHERE tmdb_id IN ({placeholders})",
        ALL_TMDB_IDS,
    ).fetchall()
}

db.close()


def build_movie_input(tmdb_id: int) -> MovieInputData | None:
    tmdb = tmdb_rows.get(tmdb_id)
    if tmdb is None:
        print(f"WARNING: tmdb_id {tmdb_id} not found in tmdb_data, skipping")
        return None
    imdb = imdb_rows.get(tmdb_id, {})

    release_date = tmdb["release_date"] or ""
    release_year = int(release_date[:4]) if release_date else None

    # Prefer IMDB maturity_rating; fall back to TMDB where IMDB has none.
    maturity_rating = imdb.get("maturity_rating") or tmdb["maturity_rating"] or ""

    return MovieInputData(
        tmdb_id=tmdb_id,
        title=tmdb["title"],
        release_year=release_year,
        overview=imdb.get("overview") or "",
        genres=imdb.get("genres") or [],
        plot_synopses=imdb.get("synopses") or [],      # imdb_data column is "synopses"
        plot_summaries=imdb.get("plot_summaries") or [],
        plot_keywords=imdb.get("plot_keywords") or [],
        overall_keywords=imdb.get("overall_keywords") or [],
        featured_reviews=imdb.get("featured_reviews") or [],
        reception_summary=imdb.get("reception_summary"),
        audience_reception_attributes=[],              # not stored in the DB
        maturity_rating=maturity_rating,
        maturity_reasoning=imdb.get("maturity_reasoning") or [],
        parental_guide_items=imdb.get("parental_guide_items") or [],
    )


movies = [m for tmdb_id in ALL_TMDB_IDS if (m := build_movie_input(tmdb_id)) is not None]
print(f"Built {len(movies)} MovieInputData objects from {len(ALL_TMDB_IDS)} requested IDs")

Built 70 MovieInputData objects from 70 requested IDs


In [23]:
from implementation.llms.generic_methods import LLMProvider

# Each function wraps generate_plot_events with a specific provider/model config.
# None are called here — invoke them individually in cells below.

async def run_gpt5_mini(movie: MovieInputData):
    """OpenAI gpt-5-mini, minimal reasoning, low verbosity."""
    return await generate_plot_events(
        movie,
        provider=LLMProvider.OPENAI,
        model="gpt-5-mini",
        reasoning_effort="minimal",
        verbosity="low",
    )

async def run_gpt5_nano(movie: MovieInputData):
    """OpenAI gpt-5-nano, minimal reasoning, low verbosity."""
    return await generate_plot_events(
        movie,
        provider=LLMProvider.OPENAI,
        model="gpt-5-nano",
        reasoning_effort="minimal",
        verbosity="low",
    )

async def run_qwen35_flash(movie: MovieInputData):
    """Alibaba Qwen 3.5 Flash via DashScope. No reasoning_effort param;
    temperature=0 gives the most deterministic/minimal output."""
    return await generate_plot_events(
        movie,
        provider=LLMProvider.ALIBABA,
        model="qwen3.5-flash",  # verify exact DashScope model ID
        temperature=0.2,
        extra_body={"enable_thinking": False},
    )

async def run_gemini25_flash(movie: MovieInputData):
    """Gemini 2.5 Flash. thinking_budget=0 disables the thinking phase
    (lowest reasoning cost)."""
    return await generate_plot_events(
        movie,
        provider=LLMProvider.GEMINI,
        model="gemini-2.5-flash",
        thinking_config={"thinking_budget": 0},
    )

async def run_gemini25_flash_lite(movie: MovieInputData):
    """Gemini 2.5 Flash Lite. thinking_budget=0 disables thinking."""
    return await generate_plot_events(
        movie,
        provider=LLMProvider.GEMINI,
        model="gemini-2.5-flash-lite",  # verify exact model ID
        thinking_config={"thinking_budget": 0},
    )

async def run_gpt_oss_120b(movie: MovieInputData):
    """GPT OSS 120B via OpenAI, minimal reasoning, low verbosity."""
    return await generate_plot_events(
        movie,
        provider=LLMProvider.GROQ,
        model="openai/gpt-oss-120b",  # verify exact OpenAI model ID
        reasoning_effort="low",
        reasoning_format="hidden",
    )

async def run_llama4_maverick(movie: MovieInputData):
    """Llama 4 Maverick via Groq. No reasoning_effort param;
    temperature=0 gives most deterministic output."""
    return await generate_plot_events(
        movie,
        provider=LLMProvider.GROQ,
        model="meta-llama/llama-4-scout-17b-16e-instruct",  # verify exact Groq model ID
        temperature=0.2,
    )

In [ ]:
first_movie = movies[54]

# ── Movie data summary ──────────────────────────────────────────────────────
print(f"Title/year:       {first_movie.title_with_year()}")
print(f"Overview:         {first_movie.overview}")

first_synopsis = first_movie.plot_synopses[0] if first_movie.plot_synopses else None
print(f"Synopsis (first): {first_synopsis[:100] if first_synopsis else '(none)'}")

print(f"Plot summaries:   {len(first_movie.plot_summaries)} entries")
for i, summary in enumerate(first_movie.plot_summaries):
    print(f"  [{i}] {summary[:100]}")

print(f"Plot keywords:    {first_movie.plot_keywords}")

# ── Run each model ───────────────────────────────────────────────────────────
RUNNERS = [
    # ("gpt-5-mini",                                         run_gpt5_mini),
    # ("gpt-5-nano",                                         run_gpt5_nano),
    # ("qwen3.5-flash",                                      run_qwen35_flash),
    # ("gemini-2.5-flash",                                   run_gemini25_flash),
    # ("gemini-2.5-flash-lite",                              run_gemini25_flash_lite),
    # ("gpt-oss-120b",                run_gpt_oss_120b),
    # ("llama-4-scout",            run_llama4_maverick),
]

for label, runner in RUNNERS:
    print(f"\n{'─' * 60}")
    print(f"Model: {label}")
    try:
        result, token_usage = await runner(first_movie)
        print(f"Result:       {result}")
        print(f"Token usage:  {token_usage}")
    except Exception as e:
        print(f"ERROR: {e}")

Title/year:       The Arrival of a Train at La Ciotat (1896)
Overview:         A train arrives at La Ciotat station.
Synopsis (first): (none)
Plot summaries:   2 entries
  [0] A group of people are standing in a straight line along the platform of a railway station, waiting f
  [1] Likely in June 1897, a group of people are standing along the platform of a railway station in La Ci
Plot keywords:    ['actuality film', 'year 1896', 'train', 'train station', '1890s', '19th century', 'train platform', 'passenger', 'silent film', 'french film', 'lumiere brothers film', 'france', 'early cinema', 'black and white movie', 'stationary camera']

────────────────────────────────────────────────────────────
Model: gpt-5-nano
user_prompt: 
title: The Arrival of a Train at La Ciotat (1896)
overview: A train arrives at La Ciotat station.
plot_summaries: 
- A group of people are standing in a straight line along the platform of a railway station, waiting for a train, which is seen coming at some dista

In [5]:
input_cost_pm = 0.125
output_cost_pm = 1
input_tokens_per_movie = 2737
output_tokens_per_movie = 953
total_movie_count = 112_000


input_cost = input_tokens_per_movie * total_movie_count * (input_cost_pm / 1_000_000)
output_cost = output_tokens_per_movie * total_movie_count * (output_cost_pm / 1_000_000)

total_cost = input_cost + output_cost

print(f"Total cost: ${total_cost:.2f}")


Total cost: $145.05
